In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("KGramConstruction") \
    .getOrCreate()

sc = spark.sparkContext

26/03/01 18:24:02 WARN Utils: Your hostname, rajesh-pc resolves to a loopback address: 127.0.1.1; using 192.168.0.39 instead (on interface wlp1s0)
26/03/01 18:24:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 18:24:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/01 18:24:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
import random
import hashlib
from itertools import combinations

In [3]:
base_path = "/home/rajesh/CSL7100/Assignment2/minhash/"

files = [
    base_path + "D1.txt",
    base_path + "D2.txt",
    base_path + "D3.txt",
    base_path + "D4.txt"
]

In [4]:
docs = {}

for i, file in enumerate(files):
    with open(file, "r") as f:
        docs[f"D{i+1}"] = f.read().strip()


In [5]:
print("D1: " + docs["D1"][:50])
print("D2: " + docs["D2"][:50])
print("D3: " + docs["D3"][:50])
print("D4: " + docs["D4"][:50])


D1: apple ceo tim cook is spending some time in canada
D2: apple ceo tim cook is spending some time in canada
D3: as part of his one day tour of canada yesterday ti
D4: president trump who warned as a candidate about th


## BUILD ALL k-GRAMS ONCE (Question 1)

In [6]:
def char_kgrams(text, k):
    text = text.strip()        # remove leading/trailing spaces
    result = set()             # use set to remove duplicates

    for i in range(len(text) - k + 1):   # valid start positions
        gram = text[i:i+k]               # substring of length k
        result.add(gram)                 # add to set (no duplicates)

    return result

In [7]:
def word_2grams(text):
    words = text.strip().split()   # split into words
    result = set()                 # use set to remove duplicates

    for i in range(len(words) - 1):   # until second-last word
        gram = words[i] + " " + words[i+1]   # combine two words
        result.add(gram)                   # add to set

    return result

In [8]:
char2_sets = {}
char3_sets = {}
word2_sets = {}

for name, text in docs.items():

    # Character 2-grams (set removes duplicates)
    char2_sets[name] = char_kgrams(text,2)

    # Character 3-grams
    char3_sets[name] = char_kgrams(text,3)

    # Word 2-grams
    words = text.split()
    word2_sets[name] = word_2grams(text)

In [10]:
print(list(char2_sets["D1"])[:5])
print(list(char3_sets["D2"])[:5])
print(list(word2_sets["D2"])[:5])

[' w', 'ne', ' a', 'mi', 'at']
['ros', 'tin', 'anc', 'ser', 'oks']
['not cook', 'all of', 'experience was', 'compete with', 'speaker from']


In [11]:
# Use already created 3-gram sets
S1 = char3_sets["D1"]
S2 = char3_sets["D2"]

In [15]:
import random
import time

m = 20011   # large prime > 10000

# Generate t hash functions of form (a*x + b) % m. Ths will return list of t -tuples (a,b)  for t differnt hash functions
def generate_hash_functions(t):
    funcs = []
    for _ in range(t):
        a = random.randint(1, m-1)   # random multiplier
        b = random.randint(0, m-1)   # random offset
        funcs.append((a, b))
    return funcs


# Compute MinHash signature for a set of shingles for the set of t different hash functions.
def compute_signature(shingle_set, hash_funcs):
    signature = []
    # for a element in shingles set find the minimun hash value for a particular has functions. this will done for t - hash functions.
    for a, b in hash_funcs:
        min_hash = float('inf')   # initialize minimum
        for sh in shingle_set:
            x = hash(sh) % m      # convert shingle to integer
            h = (a * x + b) % m   # apply hash function for the family of hash functions
            if h < min_hash:
                min_hash = h      # update minimum
        signature.append(min_hash)
    return signature


# Estimate similarity from signatures
def estimate_jaccard(sig1, sig2):
    matches = sum(1 for i in range(len(sig1)) if sig1[i] == sig2[i])  # count matches
    return matches / len(sig1)


# True Jaccard similarity
def true_jaccard(set1, set2):
    return len(set1 & set2) / len(set1 | set2)  # exact similarity




In [16]:

# Use already created 3-gram sets
S1 = char3_sets["D1"]
S2 = char3_sets["D2"]

t_values = [20, 60, 150, 300, 600]

for t in t_values:
    start = time.time()   # start timer
    
    hash_funcs = generate_hash_functions(t)   # generate t hashes
    
    sig1 = compute_signature(S1, hash_funcs)  # signature for D1
    sig2 = compute_signature(S2, hash_funcs)  # signature for D2
    
    approx = estimate_jaccard(sig1, sig2)     # estimated similarity
    
    end = time.time()   # end timer
    
    print(f"t={t}, Approx Jaccard={approx:.4f}, Time={end-start:.4f}s")

t=20, Approx Jaccard=0.9500, Time=0.0059s
t=60, Approx Jaccard=1.0000, Time=0.0108s
t=150, Approx Jaccard=0.9733, Time=0.0405s
t=300, Approx Jaccard=0.9800, Time=0.0813s
t=600, Approx Jaccard=0.9867, Time=0.1063s


In [17]:
# Compute true Jaccard similarity between two sets
def true_jaccard(set1, set2):
    intersection = set1 & set2   # common elements
    union = set1 | set2          # all unique elements
    return len(intersection) / len(union)   # J(A,B) = |A∩B| / |A∪B|

# Compute and print actual similarity
actual_similarity = true_jaccard(S1, S2)
print("Actual Jaccard Similarity (D1, D2):", actual_similarity)

Actual Jaccard Similarity (D1, D2): 0.977979274611399


In [58]:
if spark:
    spark.stop()